# Logs Transformation

In [1]:
# Installing modules

import re
import pandas as pd
import io

In [2]:
#With that 1 sample file we are gonna do transformation

with open("support_logs_2025-07-01.log", encoding='utf-8') as f:       #encoding='utf-8' is character encoding( supports all languages ,symbols and emojis)
    content = f.read()             
len(content)
# print(content)

32938

In [6]:
# list comprehension 
entries = [entry.strip() for entry in content.split("---") if entry.strip()]    
# entries[-1]     #last entry

# content.split("---")       #To split each logs

entries

# This splits the text wherever "---" appears.
# entry.strip()
# This removes extra spaces, tabs, newlines from each chunk.
# ["apple", " ", "", "banana"]
# strip() will turn " " and "" into "", but they are still items.
# ["apple", "", "banana"]
# Only keep this entry if it is NOT empty after stripping.
# ["apple","banana"]

['2025-07-01 00:21:00 [INF0] careplus.support.GenericService - TicketID=TCK0701000 SessionID=sess_TCK0701000\nIP=60.130.155.7 | ResponseTime=1269ms | CPU=27.64% | EventType=generic_event | Error=false\nUserAgent="PostmanRuntime/7.32.2"\nMessage=" event for TCK0701000"\nDebug="ℹ️ Logged for monitoring"\nTraceID=None',
 '2025-07-01 00:41:00 [INFO] careplus.support.GenericService - TicketID=TCK0701000 SessionID=sess_TCK0701000\nIP=58.36.189.27 | ResponseTime=1505ms | CPU=57.24% | EventType=generic_event | Error=false\nUserAgent="Mobile-Safari/537.36"\nMessage=" event for TCK0701000"\nDebug="ℹ️ Logged for monitoring"\nTraceID=None',
 '2025-07-01 01:44:00 [DEBUG] careplus.support.GenericService - TicketID=TCK0701001 SessionID=sess_TCK0701001\nIP=181.18.12.170 | ResponseTime=586ms | CPU=78.43% | EventType=generic_event | Error=false\nUserAgent="curl/7.68.0"\nMessage=" event for TCK0701001"\nDebug="ℹ️ Logged for monitoring"\nTraceID=None',
 '2025-07-01 01:49:00 [DEBUG] careplus.support.Generi

In [8]:
# Regex pattern to extract data
log_pattern = re.compile(
    r'(?P<timestamp>\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}) \[(?P<log_level>[A-Za-z0-9_]+)\] '
    r'(?P<component>[^\s]+) - TicketID=(?P<ticket_id>[^\s]+) SessionID=(?P<session_id>[^\s]+)\s*'
    r'IP=(?P<ip>.*?) \| ResponseTime=(?P<response_time>-?\d+)ms \| CPU=(?P<cpu>[\d.]+)% \| EventType=(?P<event_type>.*?) \| Error=(?P<error>\w+)\s*'
    r'UserAgent="(?P<user_agent>.*?)"\s*'
    r'Message="(?P<message>.*?)"\s*'
    r'Debug="(?P<debug>.*?)"\s*'
    r'TraceID=(?P<trace_id>.*)'
)

# Extract structured data
parsed_entries = []
for entry in entries:
    match = log_pattern.search(entry)
    if match:
        parsed_entries.append(match.groupdict())
        
parsed_entries[-1]      


# Loops through each log entry

# Applies a regex pattern (log_pattern.search(entry))

# If the regex matches → extracts the captured groups

# Converts them into a dictionary using groupdict()

# Appends that dictionary to parsed_entries

# Finally, parsed_entries[-1] returns the last parsed log entry

{'timestamp': '2025-07-01 14:10:00',
 'log_level': 'INFO',
 'component': 'careplus.support.GenericService',
 'ticket_id': 'TCK0701029',
 'session_id': 'sess_TCK0701029',
 'ip': '178.77.232.8',
 'response_time': '214',
 'cpu': '33.47',
 'event_type': 'generic_event',
 'error': 'false',
 'user_agent': 'curl/7.68.0',
 'message': ' event for TCK0701029',
 'debug': 'ℹ️ Logged for monitoring',
 'trace_id': 'None'}

In [12]:
#Tranforming the regex pattern to dataframe

df = pd.DataFrame(parsed_entries)
df.head(3)

,timestamp,log_level,component,ticket_id,session_id,ip,response_time,cpu,event_type,error,user_agent,message,debug,trace_id
0,2025-07-01 00:21:00,INF0,careplus.support.GenericService,TCK0701000,sess_TCK0701000,60.130.155.7,1269,27.64,generic_event,false,PostmanRuntime/7.32.2,event for TCK0701000,ℹ️ Logged for monitoring,None
1,2025-07-01 00:41:00,INFO,careplus.support.GenericService,TCK0701000,sess_TCK0701000,58.36.189.27,1505,57.24,generic_event,false,Mobile-Safari/537.36,event for TCK0701000,ℹ️ Logged for monitoring,None
2,2025-07-01 01:44:00,DEBUG,careplus.support.GenericService,TCK0701001,sess_TCK0701001,181.18.12.170,586,78.43,generic_event,false,curl/7.68.0,event for TCK0701001,ℹ️ Logged for monitoring,None


In [13]:
df = df.drop(columns="trace_id")       #Dropping traceid none column
df.head(2)

# You give axis=1 in drop() when you want to drop columns.
# You give axis=0 (or nothing) when you want to drop rows.

# To avoid this we can say You can also drop columns using columns=:df.drop(columns='age')

,timestamp,log_level,component,ticket_id,session_id,ip,response_time,cpu,event_type,error,user_agent,message,debug
0,2025-07-01 00:21:00,INF0,careplus.support.GenericService,TCK0701000,sess_TCK0701000,60.130.155.7,1269,27.64,generic_event,false,PostmanRuntime/7.32.2,event for TCK0701000,ℹ️ Logged for monitoring
1,2025-07-01 00:41:00,INFO,careplus.support.GenericService,TCK0701000,sess_TCK0701000,58.36.189.27,1505,57.24,generic_event,false,Mobile-Safari/537.36,event for TCK0701000,ℹ️ Logged for monitoring


In [15]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 105 entries, 0 to 104
Data columns (total 13 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   timestamp      105 non-null    object
 1   log_level      105 non-null    object
 2   component      105 non-null    object
 3   ticket_id      105 non-null    object
 4   session_id     105 non-null    object
 5   ip             105 non-null    object
 6   response_time  105 non-null    object
 7   cpu            105 non-null    object
 8   event_type     105 non-null    object
 9   error          105 non-null    object
 10  user_agent     105 non-null    object
 11  message        105 non-null    object
 12  debug          105 non-null    object
dtypes: object(13)
memory usage: 10.8+ KB


In [14]:
#Conveting columns to responding data type

df = df.astype({                  #astype for typecasting
    "response_time": "int",
    "cpu": "float"
})
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 105 entries, 0 to 104
Data columns (total 13 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   timestamp      105 non-null    object 
 1   log_level      105 non-null    object 
 2   component      105 non-null    object 
 3   ticket_id      105 non-null    object 
 4   session_id     105 non-null    object 
 5   ip             105 non-null    object 
 6   response_time  105 non-null    int64  
 7   cpu            105 non-null    float64
 8   event_type     105 non-null    object 
 9   error          105 non-null    object 
 10  user_agent     105 non-null    object 
 11  message        105 non-null    object 
 12  debug          105 non-null    object 
dtypes: float64(1), int64(1), object(11)
memory usage: 10.8+ KB


In [ ]:
df['error'] = df['error'].str.lower().map({'true': True, 'false': False})      #Error to boolean
df['timestamp'] = pd.to_datetime(df['timestamp'], format='%Y-%m-%d %H:%M:%S', errors='coerce').astype('datetime64[ms]')    #in timestamp format

In [18]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 105 entries, 0 to 104
Data columns (total 13 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   timestamp      105 non-null    datetime64[ms]
 1   log_level      105 non-null    object        
 2   component      105 non-null    object        
 3   ticket_id      105 non-null    object        
 4   session_id     105 non-null    object        
 5   ip             105 non-null    object        
 6   response_time  105 non-null    int32         
 7   cpu            105 non-null    float64       
 8   event_type     105 non-null    object        
 9   error          105 non-null    bool          
 10  user_agent     105 non-null    object        
 11  message        105 non-null    object        
 12  debug          105 non-null    object        
dtypes: bool(1), datetime64[ms](1), float64(1), int32(1), object(9)
memory usage: 9.7+ KB


In [ ]:
df.describe()     #des the  numerical colum in statistical way

,timestamp,response_time,cpu
count,105,105.000000,105.000000
mean,2025-07-01 08:21:31.428000,885.523810,54.918190
min,2025-07-01 00:21:00,-1566.000000,10.140000
25%,2025-07-01 05:38:00,586.000000,34.420000
50%,2025-07-01 09:13:00,984.000000,60.140000
75%,2025-07-01 11:12:00,1323.000000,73.700000
max,2025-07-01 14:10:00,1792.000000,89.970000
std,NaN,640.849858,22.513155


In [15]:
df = df[df.response_time>=0]      #removing -ve values
df.describe()

,response_time,cpu
count,98.000000,98.000000
mean,1006.500000,54.821633
std,447.808944,22.185337
min,126.000000,13.350000
25%,653.250000,34.710000
50%,1089.000000,59.455000
75%,1327.750000,73.242500
max,1792.000000,89.970000


In [ ]:
df["log_level"].value_counts()   

log_level
INFO       37
DEBUG      32
INF0       13
DEBG       10
warnING     3
WARNING     3
Name: count, dtype: int64

In [16]:
fix_log_level = {'INF0': 'INFO', 'DEBG': 'DEBUG', 'warnING': 'WARNING', 'EROR': 'ERROR'}    #correcting the spells
df['log_level'] = df['log_level'].replace(fix_log_level)
    
df.log_level.value_counts()

log_level
INFO       50
DEBUG      42
WARNING     6
Name: count, dtype: int64

In [ ]:
df[df.duplicated()]              #pandas detect duplicates only if entire row is duplicated

,timestamp,log_level,component,ticket_id,session_id,ip,response_time,cpu,event_type,error,user_agent,message,debug
10,2025-07-01 03:30:00,DEBUG,careplus.support.GenericService,TCK0701002,sess_TCK0701002,36.64.191.144,1223,81.83,generic_event,false,curl/7.68.0,event for TCK0701002,ℹ️ Logged for monitoring
33,2025-07-01 06:26:00,INFO,careplus.support.GenericService,TCK0701009,sess_TCK0701009,30.228.28.191,1253,32.54,generic_event,false,PostmanRuntime/7.32.2,event for TCK0701009,ℹ️ Logged for monitoring
38,2025-07-01 06:43:00,DEBUG,careplus.support.GenericService,TCK0701008,sess_TCK0701008,214.140.181.78,1372,63.43,generic_event,false,Mobile-Safari/537.36,event for TCK0701008,ℹ️ Logged for monitoring
44,2025-07-01 07:57:00,INFO,careplus.support.GenericService,TCK0701018,sess_TCK0701018,167.18.200.246,1454,85.97,generic_event,false,Mozilla/5.0 (Windows NT 10.0),event for TCK0701018,ℹ️ Logged for monitoring
57,2025-07-01 09:32:00,DEBUG,careplus.support.GenericService,TCK0701013,sess_TCK0701013,114.173.55.131,1089,32.70,generic_event,false,PostmanRuntime/7.32.2,event for TCK0701013,ℹ️ Logged for monitoring
59,2025-07-01 09:36:00,DEBUG,careplus.support.GenericService,TCK0701020,sess_TCK0701020,146.157.172.98,808,78.08,generic_event,false,Mozilla/5.0 (Windows NT 10.0),event for TCK0701020,ℹ️ Logged for monitoring
69,2025-07-01 10:17:00,INFO,careplus.support.GenericService,TCK0701019,sess_TCK0701019,163.229.213.118,1568,24.09,generic_event,false,PostmanRuntime/7.32.2,event for TCK0701019,ℹ️ Logged for monitoring
77,2025-07-01 10:45:00,INFO,careplus.support.GenericService,TCK0701023,sess_TCK0701023,30.211.17.103,1127,22.14,generic_event,false,Mozilla/5.0 (Windows NT 10.0),event for TCK0701023,ℹ️ Logged for monitoring
92,2025-07-01 12:25:00,DEBUG,careplus.support.GenericService,TCK0701028,sess_TCK0701028,138.124.89.86,1250,46.43,generic_event,false,Mobile-Safari/537.36,event for TCK0701028,ℹ️ Logged for monitoring


In [18]:
df = df.drop_duplicates()
df[df.duplicated()]

,timestamp,log_level,component,ticket_id,session_id,ip,response_time,cpu,event_type,error,user_agent,message,debug


In [19]:
df.shape

(89, 13)